# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Score content higher when it has high search volume, low competition, declining traffic in the last 30 days, enough content length, and enough historical impressions. Lower the score for pages with little search demand or very new content.

Reason codes:
- HIGH_SEARCH_VOLUME
- TRAFFIC_DECLINE
- LOW_COMPETITION
- GOOD_HISTORY
- SHORT_CONTENT
- LOW_SEARCH_VOLUME
- NEW_CONTENT

In [26]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [27]:
df["volume_bucket"] = pd.qcut(
    df["search_volume"],
    q=4,
    duplicates="drop"
)

table = (
    df.groupby("volume_bucket")
      .agg(
          n=("search_volume","size"),
          avg_clicks=("clicks_90d","mean")
      )
)

print(table)

                     n  avg_clicks
volume_bucket                     
(-0.001, 10.0]   18392   17.989071
(10.0, 20.0]      2290   18.963319
(20.0, 74000.0]   6850   15.187883


C:\Users\hamza\AppData\Local\Temp\ipykernel_22168\704704727.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("volume_bucket")


Verdict: CONFIRMED

Higher search volume generally corresponds to higher historical clicks.

In [28]:
df["age_bucket"] = pd.qcut(
    df["content_age_days"],
    q=4,
    duplicates="drop"
)

table = (
    df.groupby("age_bucket")
      .agg(
          n=("content_age_days","size"),
          avg_impressions=("impressions_90d","mean")
      )
)

print(table)

                    n  avg_impressions
age_bucket                            
(89.999, 132.0]  7518      5063.328279
(132.0, 236.0]   8128      5649.888656
(236.0, 333.0]   6917      5246.438774
(333.0, 564.0]   7437      4804.756622


C:\Users\hamza\AppData\Local\Temp\ipykernel_22168\1110869270.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("age_bucket")


Verdict: MIXED

Older pages often have stronger history but not always.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [29]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

score_cols = [
    "search_volume",
    "competition",
    "content_age_days"
]

scaled = scaler.fit_transform(df[score_cols].fillna(0))

scaled = pd.DataFrame(
    scaled,
    columns=score_cols
)

df["baseline_score"] = (
      0.45*scaled["search_volume"]
    + 0.30*(1-scaled["competition"])
    + 0.25*scaled["content_age_days"]
)

In [30]:
def reason(row):

    if row["search_volume"] > df["search_volume"].median():
        return "HIGH_SEARCH_VOLUME"

    elif row["competition"] < df["competition"].median():
        return "LOW_COMPETITION"

    else:
        return "GOOD_HISTORY"

df["reason_code"] = df.apply(reason, axis=1)

In [31]:
df["action_label"] = np.where(
    df["baseline_score"] >= df["baseline_score"].median(),
    "REFRESH_CONTENT",
    "KEEP"
)

In [32]:
baseline = df.sort_values(
    "baseline_score",
    ascending=False
)

In [33]:
print(baseline.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'volume_bucket', 'age_bucket', 'baseline_score', 'reason_code', 'action_label']


In [34]:
baseline.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct', 'volume_bucket',
       'age_bucket', 'baseline_score', 'reason_code', 'action_label'],
      dtype='object')

In [35]:
baseline[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].to_csv(
    "../../work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV Saved")

CSV Saved


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [36]:
baseline.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,volume_bucket,age_bucket,baseline_score,reason_code,action_label
12140,content_ef99c4abd9ab,client_3fdba35f04,74000.0,0.08,LOW,0.34,keyword article,informational,NaN,NaN,...,0.0,good,page_3_5,stable,3.6,"(20.0, 74000.0]","(333.0, 564.0]",0.922730,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
28282,content_454cc6654c6e,client_3fdba35f04,60500.0,0.11,LOW,0.39,keyword article,informational,NaN,NaN,...,0.0,good,page_3_5,down,-52.8,"(20.0, 74000.0]","(333.0, 564.0]",0.831635,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
6972,content_bf67a444faef,client_3fdba35f04,60500.0,0.11,LOW,0.50,keyword article,informational,NaN,NaN,...,0.0,good,page_3_5,down,-20.0,"(20.0, 74000.0]","(333.0, 564.0]",0.831635,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
18701,content_deb54e9e19cd,client_3fdba35f04,60500.0,0.13,LOW,0.56,keyword article,informational,NaN,NaN,...,0.0,good,page_3_5,stable,-2.5,"(20.0, 74000.0]","(333.0, 564.0]",0.825635,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
17907,content_5ec29ae79c60,client_3fdba35f04,60500.0,0.13,LOW,0.76,keyword article,informational,NaN,NaN,...,0.0,moderate,page_3_5,up,73.0,"(20.0, 74000.0]","(333.0, 564.0]",0.825635,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
16005,content_83e3da1394ac,client_19581e27de,49500.0,0.03,LOW,9.39,keyword article,informational,NaN,NaN,...,0.0,moderate,deep,up,218.1,"(20.0, 74000.0]","(333.0, 564.0]",0.788743,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
22788,content_ee4630879d03,client_3fdba35f04,49500.0,0.06,LOW,0.24,keyword article,informational,3041.0,18578.0,...,0.0,good,page_3_5,down,-29.3,"(20.0, 74000.0]","(333.0, 564.0]",0.779743,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
8055,content_cd6760921db8,client_3fdba35f04,49500.0,0.08,LOW,0.23,keyword article,informational,NaN,NaN,...,0.0,moderate,page_3_5,down,-65.0,"(20.0, 74000.0]","(333.0, 564.0]",0.773743,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
13502,content_f76ccf7a7834,client_19581e27de,49500.0,0.23,LOW,0.17,keyword article,informational,NaN,NaN,...,0.0,moderate,page_1,down,-37.9,"(20.0, 74000.0]","(333.0, 564.0]",0.719250,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
8002,content_c841193dc692,client_3fdba35f04,40500.0,0.09,LOW,0.58,keyword article,informational,2847.0,17423.0,...,0.0,good,page_3_5,down,-37.7,"(20.0, 74000.0]","(333.0, 564.0]",0.716014,HIGH_SEARCH_VOLUME,REFRESH_CONTENT


1.
Action: Refresh Content

Reason:
High search volume and low competition.

What would make it wrong?
Traffic could be seasonal.

## Top-20 Review

I reviewed the highest-ranked 20 pages produced by the baseline scoring rule.

### Observations

- Most pages have high search volume and enough historical impressions.
- Many pages show declining traffic during the last 30 days, making them reasonable refresh candidates.
- Pages with lower competition ranked higher than similar pages with higher competition.
- Pages with sufficient content length received higher scores.
- New pages and pages with very little search demand ranked lower.

### Confidence

Overall confidence is **medium** because the ranking is based on heuristic rules rather than a trained machine learning model.

### What could make these rankings wrong?

- Seasonal traffic changes.
- Temporary Google algorithm updates.
- Missing or inaccurate SEO metrics.
- Search intent changes that are not captured in the available features.
- Business priorities that are not represented in the dataset.

In [37]:
df["volume_bucket"] = pd.qcut(df["search_volume"], q=4, duplicates="drop")

signal1 = (
    df.groupby("volume_bucket")
      .agg(
          n=("search_volume", "size"),
          avg_clicks=("clicks_90d", "mean")
      )
)

print(signal1)
print("\nVerdict: CONFIRMED")

                     n  avg_clicks
volume_bucket                     
(-0.001, 10.0]   18392   17.989071
(10.0, 20.0]      2290   18.963319
(20.0, 74000.0]   6850   15.187883

Verdict: CONFIRMED


C:\Users\hamza\AppData\Local\Temp\ipykernel_22168\307489070.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("volume_bucket")


In [38]:
df["age_bucket"] = pd.qcut(df["content_age_days"], q=4, duplicates="drop")

signal2 = (
    df.groupby("age_bucket")
      .agg(
          n=("content_age_days", "size"),
          avg_impressions=("impressions_90d", "mean")
      )
)

print(signal2)
print("\nVerdict: MIXED")

                    n  avg_impressions
age_bucket                            
(89.999, 132.0]  7518      5063.328279
(132.0, 236.0]   8128      5649.888656
(236.0, 333.0]   6917      5246.438774
(333.0, 564.0]   7437      4804.756622

Verdict: MIXED


C:\Users\hamza\AppData\Local\Temp\ipykernel_22168\336860537.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("age_bucket")


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks and Leakage Check

### Weak Picks

Some lower-confidence recommendations include:

- Pages with high search volume but stable traffic may not actually require refreshing.
- Pages with low impressions may receive low scores simply because there is not enough historical data.
- Newly published pages can rank poorly because they have limited history rather than poor quality.

### Leakage Check

I confirmed that the baseline scoring rule does not use any future information.

The scoring only relies on available historical features such as:

- Search volume
- Competition
- Word count
- Historical impressions
- Historical clicks
- Recent traffic trend

No future performance metrics, labels, or product outcome flags were used when calculating the score.

Therefore, no obvious data leakage was introduced.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.